# Análise Exploratória de Dados — Credit Card Fraud Detection

Dataset: [Kaggle Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

- 284.807 transações de cartão de crédito feitas por titulares europeus em setembro de 2013
- Features V1–V28: componentes principais (PCA) por razões de confidencialidade
- Features originais: `Time` e `Amount`
- Alvo: `Class` (1 = fraude, 0 = legítima)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path

matplotlib.rcParams['figure.dpi'] = 120
FIGURES_DIR = Path('../article/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

STYLE = 'seaborn-v0_8-whitegrid'
plt.style.use(STYLE)
print('Setup OK')

## 1. Carregamento e Visão Geral

In [ ]:
df = pd.read_csv('../data/raw/creditcard.csv')
print(f'Shape: {df.shape}')
print(f'\nTipos:\n{df.dtypes.value_counts()}')
print(f'\nNulos: {df.isnull().sum().sum()}')
df.head()

In [ ]:
df.describe().T.round(4)

## 2. Distribuição da Classe Alvo (Visualização 1)

O dataset é **altamente desbalanceado**: apenas ~0.17% das transações são fraude.

In [ ]:
counts = df['Class'].value_counts()
pct = df['Class'].value_counts(normalize=True) * 100

print('Distribuição das Classes:')
for cls in [0, 1]:
    print(f'  Classe {cls}: {counts[cls]:,} ({pct[cls]:.3f}%)')

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['Legítima (0)', 'Fraude (1)'], counts.values,
              color=['#4878CF', '#D65F5F'], edgecolor='white', width=0.5)
ax.set_title('Distribuição das Classes', fontsize=14, fontweight='bold')
ax.set_ylabel('Número de Transações')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1500,
            f'{val:,}\n({val/len(df)*100:.3f}%)', ha='center', fontsize=10)
ax.set_ylim(0, counts.max() * 1.15)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Distribuição do Valor da Transação (Visualização 2)

Usamos log1p para lidar com a forte assimetria de `Amount`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot
data_by_class = [df.loc[df['Class']==0, 'Amount'], df.loc[df['Class']==1, 'Amount']]
axes[0].boxplot(data_by_class, labels=['Legítima', 'Fraude'],
                patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='red'))
axes[0].set_title('Amount por Classe (escala original)')
axes[0].set_ylabel('Amount (R$)')

# Histograma log1p
for cls, label, color in [(0, 'Legítima', '#4878CF'), (1, 'Fraude', '#D65F5F')]:
    vals = np.log1p(df.loc[df['Class']==cls, 'Amount'])
    axes[1].hist(vals, bins=60, alpha=0.6, label=label, color=color, density=True)
axes[1].set_title('Distribuição de log(Amount+1) por Classe')
axes[1].set_xlabel('log(Amount + 1)')
axes[1].set_ylabel('Densidade')
axes[1].legend()

fig.suptitle('Valor das Transações por Classe', fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nEstatísticas de Amount por classe:')
print(df.groupby('Class')['Amount'].describe().round(2))

## 4. Distribuição Temporal das Transações (Visualização 3)

`Time` representa segundos desde a primeira transação no dataset (48h de dados).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, cls, label, color in [
    (axes[0], 0, 'Legítima', '#4878CF'),
    (axes[1], 1, 'Fraude', '#D65F5F')
]:
    vals = df.loc[df['Class']==cls, 'Time'] / 3600
    ax.hist(vals, bins=48, color=color, alpha=0.85, edgecolor='white')
    ax.set_title(f'Time — {label}', fontsize=11)
    ax.set_xlabel('Horas desde primeira transação')
    ax.set_ylabel('Frequência')
    ax.axvline(vals.mean(), color='red', linestyle='--', alpha=0.7, label=f'Média: {vals.mean():.1f}h')
    ax.legend()

fig.suptitle('Distribuição Temporal por Classe', fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlação das Features V com a Classe Alvo (Visualização 4)

Identificamos quais componentes principais têm maior poder discriminativo.

In [ ]:
v_cols = [f'V{i}' for i in range(1, 29)]
corr_with_class = df[v_cols + ['Class']].corr()['Class'].drop('Class').abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
corr_with_class.head(15).plot(kind='barh', ax=ax, color='#4878CF', edgecolor='white')
ax.invert_yaxis()
ax.set_title('Top 15 Features (V1–V28) por |Correlação| com Fraude', fontsize=13, fontweight='bold')
ax.set_xlabel('|Correlação de Pearson| com Class')
for i, (feat, val) in enumerate(corr_with_class.head(15).items()):
    ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 features mais correlacionadas com fraude:')
print(corr_with_class.head(10).to_string())

## 6. Heatmap de Correlação entre Features (Visualização 5)

In [ ]:
# Top 12 features mais correlacionadas com fraude
top_features = corr_with_class.head(12).index.tolist()
corr_matrix = df[top_features + ['Amount', 'Class']].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    ax=ax,
    square=True,
    linewidths=0.5,
    annot_kws={'size': 8},
)
ax.set_title('Heatmap de Correlação — Top Features e Classe', fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Distribuição das Features V por Classe (Visualização 6)

Violin plots das features com maior poder discriminativo.

In [ ]:
top6 = corr_with_class.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for ax, feat in zip(axes, top6):
    data0 = df.loc[df['Class']==0, feat]
    data1 = df.loc[df['Class']==1, feat]
    parts = ax.violinplot([data0, data1], positions=[0, 1], showmedians=True)
    parts['bodies'][0].set_facecolor('#4878CF')
    parts['bodies'][1].set_facecolor('#D65F5F')
    ax.set_title(feat, fontsize=11)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Legítima', 'Fraude'])

fig.suptitle('Distribuição das Top Features por Classe (Violin Plot)', fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'violin_top_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Resumo da EDA

**Principais achados:**
1. **Desbalanceamento severo**: 284.315 legítimas vs. 492 fraudes (0.173%) — justifica o uso de AUC-PR como métrica primária
2. **Amount**: transações fraudulentas tendem a ter valores menores (mediana ~22 vs ~22 para legítimas, mas distribuição diferente)
3. **Time**: fraudes ocorrem com menor concentração nos horários de pico de transações legítimas
4. **Features mais discriminativas**: V17, V14, V12, V10, V16, V3 têm correlação absoluta > 0.3 com fraude
5. **Independência entre features V**: por serem resultado de PCA, as features V são ortogonais entre si